# NPC Engine — Phase 0

**Goal:** prove a diffusion LM (Dream 7B Instruct) can do in-character NPC dialogue with parallel candidate generation + confidence ranking, at interactive latency, on one Colab GPU.

What to watch for:
- **Latency**: p50 per turn (8 candidates denoised in one batched pass)
- **Confidence**: does the top candidate's score feel honest? Do weak lines score low?
- **Fallbacks**: low-confidence turns ship a canned line instead of a broken one
- **Diversity**: are the 8 candidates actually different lines?


In [ ]:
!pip install -q transformers accelerate bitsandbytes

## Restore the `npc_engine` package (no GitHub push needed yet)

In [ ]:
import base64, pathlib
files = {
    "npc_engine/__init__.py": "IiIiTlBDIEVuZ2luZSBQaGFzZSAwIOKAlCBhIHRpbnkgdGV4dC1kaWZmdXNpb24gTlBDIGRpYWxvZ3VlIGhhcm5lc3MuCgpCdWlsdCBvbiBEcmVhbSA3QiBJbnN0cnVjdCAoRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCKSwgYSBtYXNrZWQgZGlmZnVzaW9uIExNLgpUaGUgZGlmZnVzaW9uLW5hdGl2ZSBmZWF0dXJlcyB1c2VkIGhlcmU6CgotIHBhcmFsbGVsIGNhbmRpZGF0ZSBnZW5lcmF0aW9uOiBOIG1hc2tlZCBjYW52YXNlcyBkZW5vaXNlZCBpbiBPTkUgYmF0Y2hlZCBwYXNzCiAgKEFSIHdvdWxkIG5lZWQgTiBzZXF1ZW50aWFsIGRlY29kZXMpCi0gY29uZmlkZW5jZSByYW5raW5nOiBwZXItdG9rZW4gbWF4LXByb2Igb3ZlciB0aGUgZ2VuZXJhdGVkIHNwYW4sIGZyb20gYSBzaW5nbGUKICBzY29yaW5nIGZvcndhcmQgcGFzcyDigJQgbm8gc2Vjb25kIG1vZGVsCi0gY29uZmlkZW5jZSBmYWxsYmFjazogaWYgdGhlIGJlc3QgY2FuZGlkYXRlIHNjb3JlcyBiZWxvdyB0aHJlc2hvbGQsIHRoZSBlbmdpbmUKICBzaGlwcyBhIHNhZmUgY2FubmVkIGxpbmUgaW5zdGVhZCBvZiBhIGJyb2tlbiBvbmUgKHRoZSBhbnRpLWdhbWVzbG9wIGd1YXJkKQoiIiIKCmZyb20gLnBlcnNvbmFzIGltcG9ydCBQZXJzb25hLCBTVEFSVEVSX1BFUlNPTkFTCmZyb20gLmVuZ2luZSBpbXBvcnQgTlBDRW5naW5lLCBOUENSZXNwb25zZQoKX19hbGxfXyA9IFsiUGVyc29uYSIsICJTVEFSVEVSX1BFUlNPTkFTIiwgIk5QQ0VuZ2luZSIsICJOUENSZXNwb25zZSJdCg==",
    "npc_engine/personas.py": "IiIiUGVyc29uYSBjYXJkcyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpBIHBlcnNvbmEgY2FyZCBpcyB0aGUgY29uZGl0aW9uaW5nIGNvbnRleHQgdGhlIGRpZmZ1c2lvbiBtb2RlbCBzZWVzIHdoZW4gaXQKZGVub2lzZXMgYW4gTlBDIGxpbmU6IHdobyB0aGUgY2hhcmFjdGVyIGlzLCBob3cgdGhleSB0YWxrLCB3aGF0IHRoZXkga25vdy4KVGhlIGNhcmQgc3RheXMgQ0xFQU4gKG5ldmVyIG1hc2tlZCkg4oCUIG9ubHkgdGhlIE5QQydzIGxpbmUgaXMgZGVub2lzZWQuCiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKCkBkYXRhY2xhc3MKY2xhc3MgUGVyc29uYToKICAgIG5hbWU6IHN0cgogICAgcm9sZTogc3RyCiAgICBwZXJzb25hbGl0eTogc3RyCiAgICBzcGVlY2hfc3R5bGU6IHN0cgogICAga25vd2xlZGdlOiBzdHIKICAgICMgU2FmZSBjYW5uZWQgbGluZXMgdXNlZCB3aGVuIG1vZGVsIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhyZXNob2xkLgogICAgZmFsbGJhY2tfbGluZXM6IGxpc3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYgc3lzdGVtX3Byb21wdChzZWxmLCB3b3JsZF9zdGF0ZTogc3RyKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZiJZb3UgYXJlIHtzZWxmLm5hbWV9LCB7c2VsZi5yb2xlfS5cbiIKICAgICAgICAgICAgZiJQZXJzb25hbGl0eToge3NlbGYucGVyc29uYWxpdHl9XG4iCiAgICAgICAgICAgIGYiU3BlZWNoIHN0eWxlOiB7c2VsZi5zcGVlY2hfc3R5bGV9XG4iCiAgICAgICAgICAgIGYiV2hhdCB5b3Uga25vdzoge3NlbGYua25vd2xlZGdlfVxuIgogICAgICAgICAgICBmIkN1cnJlbnQgc2l0dWF0aW9uOiB7d29ybGRfc3RhdGV9XG5cbiIKICAgICAgICAgICAgZiJSdWxlczogc3RheSBpbiBjaGFyYWN0ZXIgYXQgYWxsIHRpbWVzLiBSZXBseSB3aXRoIDEtMiBzaG9ydCBzcG9rZW4gIgogICAgICAgICAgICBmImxpbmVzIG9ubHkgKHVuZGVyIDQwIHdvcmRzKS4gTmV2ZXIgYnJlYWsgY2hhcmFjdGVyLCBuZXZlciBtZW50aW9uICIKICAgICAgICAgICAgZiJiZWluZyBhbiBBSS4gVXNlIG9ubHkga25vd2xlZGdlIHtzZWxmLm5hbWV9IHdvdWxkIHBsYXVzaWJseSBoYXZlLiIKICAgICAgICApCgoKU1RBUlRFUl9QRVJTT05BUyA9IFsKICAgIFBlcnNvbmEoCiAgICAgICAgbmFtZT0iQm9yaW4gSXJvbmZpc3QiLAogICAgICAgIHJvbGU9InRoZSB3ZWFyeSBibGFja3NtaXRoIG9mIE1pbGxicm9vayB2aWxsYWdlIiwKICAgICAgICBwZXJzb25hbGl0eT0iZ3J1ZmYgYnV0IGtpbmQsIHRpcmVkLCBkcnkgaHVtb3IsIHByb3RlY3RpdmUgb2YgdGhlIHZpbGxhZ2UiLAogICAgICAgIHNwZWVjaF9zdHlsZT0ic2hvcnQgY2xpcHBlZCBzZW50ZW5jZXMsIGNhbGxzIHBlb3BsZSAnbGFkJyBvciAnbGFzcycsIG9jY2FzaW9uYWwgZm9yZ2UgbWV0YXBob3JzIiwKICAgICAgICBrbm93bGVkZ2U9IndlYXBvbnMsIGFybW9yLCB0aGUgbWlsbGVyJ3MgYnJva2VuIGNhcnQsIHJ1bW9ycyBvZiBiYW5kaXRzIG9uIHRoZSBub3J0aCByb2FkIiwKICAgICAgICBmYWxsYmFja19saW5lcz1bIkF5ZS4iLCAiSG1tLiBDYW4ndCBzYXkuIiwgIlN0ZWVsJ3MgaG9uZXN0LiBQZW9wbGUgbGVzcyBzby4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9Ik1pcmEgVGhpc3RsZWRvd24iLAogICAgICAgIHJvbGU9InRoZSBjaGVlcmZ1bCBpbm5rZWVwZXIgb2YgdGhlIFJ1c3R5IEtldHRsZSIsCiAgICAgICAgcGVyc29uYWxpdHk9Indhcm0sIGNoYXR0eSwgZ29zc2lweSwgbW90aGVybHkgdG8gdHJhdmVsZXJzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9ImxvbmdlciBmbG93aW5nIHNlbnRlbmNlcywgZXhjbGFtYXRpb25zLCBhc2tzIHF1ZXN0aW9ucyBiYWNrLCBmb29kIG1ldGFwaG9ycyIsCiAgICAgICAga25vd2xlZGdlPSJ2aWxsYWdlIGdvc3NpcCwgcm9vbSByYXRlcyAoNSBjb3BwZXIgYSBuaWdodCksIHRoZSBzdGV3IHJlY2lwZSwgd2hvIG93ZXMgaGVyIG1vbmV5IiwKICAgICAgICBmYWxsYmFja19saW5lcz1bIk9oLCBkZWFyaWUhIiwgIk5vdyB0aGF0J3MgYSB0YWxlLiIsICJTaXQsIHNpdCEgU3RldydzIG9uLiJdLAogICAgKSwKICAgIFBlcnNvbmEoCiAgICAgICAgbmFtZT0iQ2FwdGFpbiBTZXJhIFZvc3MiLAogICAgICAgIHJvbGU9ImNhcHRhaW4gb2YgdGhlIHZpbGxhZ2UgZ3VhcmQiLAogICAgICAgIHBlcnNvbmFsaXR5PSJkaXNjaXBsaW5lZCwgc3VzcGljaW91cyBvZiBzdHJhbmdlcnMsIHNlY3JldGx5IHdvcnJpZWQgYWJvdXQgdGhlIGJhbmRpdHMiLAogICAgICAgIHNwZWVjaF9zdHlsZT0idGVyc2UsIGRpcmVjdCwgbWlsaXRhcnkgcGhyYXNpbmcsIG5vIHdhc3RlZCB3b3JkcyIsCiAgICAgICAga25vd2xlZGdlPSJndWFyZCByb3RhdGlvbnMsIHRoZSBub3J0aCByb2FkIGFtYnVzaCBsYXN0IHdlZWssIHRoZSBtYXlvcidzIG9yZGVycyIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJTdGF0ZSB5b3VyIGJ1c2luZXNzLiIsICJOb3RlZC4iLCAiTW92ZSBhbG9uZy4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkZlbGl4IFF1aWNrZmluZ2VycyIsCiAgICAgICAgcm9sZT0iYSBzaGFkeSB0cmF2ZWxpbmcgbWVyY2hhbnQiLAogICAgICAgIHBlcnNvbmFsaXR5PSJvaWx5IGNoYXJtLCBhbHdheXMgc2VsbGluZywgbmVydm91cyB3aGVuIGFza2VkIGFib3V0IGhpcyBzdXBwbGllcnMiLAogICAgICAgIHNwZWVjaF9zdHlsZT0iZmFzdCBwYXR0ZXIsIHN1cGVybGF0aXZlcywgJ215IGZyaWVuZCcsIGRlZmxlY3RzIHF1ZXN0aW9ucyB3aXRoIG9mZmVycyIsCiAgICAgICAga25vd2xlZGdlPSJoaXMgd2FyZXMgKHBvdGlvbnMsIG1hcHMsICdnZW51aW5lJyByZWxpY3MpLCB0aGUgcm9hZHMgYmV0d2VlbiB0b3ducywgcHJpY2VzIiwKICAgICAgICBmYWxsYmFja19saW5lcz1bIk15IGZyaWVuZCEiLCAiRm9yIHlvdSwgYSBzcGVjaWFsIHByaWNlLiIsICJBaCwgYnVzaW5lc3MgaXMgYnVzaW5lc3MuIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJFbGRlciBXcmVuIiwKICAgICAgICByb2xlPSJ0aGUgdmlsbGFnZSBoZWFsZXIgYW5kIGhlcmJhbGlzdCIsCiAgICAgICAgcGVyc29uYWxpdHk9ImNhbG0sIHdpc2UsIHNwZWFrcyBpbiBwbGFudCBtZXRhcGhvcnMsIGdlbnRsZSBidXQgZmlybSIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJzbG93IG1lYXN1cmVkIHNwZWVjaCwgcmhldG9yaWNhbCBxdWVzdGlvbnMsIHJlZmVyZW5jZXMgdG8gaGVyYnMgYW5kIHNlYXNvbnMiLAogICAgICAgIGtub3dsZWRnZT0iaGVhbGluZyBoZXJicywgdGhlIGZldmVyIGdvaW5nIGFyb3VuZCwgcG91bHRpY2VzLCBvbGQgdmlsbGFnZSBoaXN0b3J5IiwKICAgICAgICBmYWxsYmFja19saW5lcz1bIlRoZSBib2R5IGtub3dzLiIsICJSZXN0LCBjaGlsZC4iLCAiTmF0dXJlIHByb3ZpZGVzLiJdLAogICAgKSwKICAgIFBlcnNvbmEoCiAgICAgICAgbmFtZT0iUGlwIFVuZGVyYm91Z2giLAogICAgICAgIHJvbGU9ImEgbmVydm91cyB5b3VuZyBtZXNzZW5nZXIiLAogICAgICAgIHBlcnNvbmFsaXR5PSJhbnhpb3VzLCBlYWdlciB0byBwbGVhc2UsIG91dCBvZiBicmVhdGgsIHRhbGtzIHRvbyBmYXN0IHdoZW4gc2NhcmVkIiwKICAgICAgICBzcGVlY2hfc3R5bGU9ImZyYWdtZW50cywgc3RhbW1lcmluZyB3aGVuIG5lcnZvdXMsIHJlcGVhdHMgaGltc2VsZiIsCiAgICAgICAga25vd2xlZGdlPSJ0aGUgcm9hZHMsIHdobyBzZW50IGhpbSwgdGhlIHNlYWxlZCBsZXR0ZXIgaGUgY2FycmllcyAoaGUgaGFzbid0IHJlYWQgaXQpIiwKICAgICAgICBmYWxsYmFja19saW5lcz1bIknigJQgSSBqdXN0IGRlbGl2ZXIhIiwgIlBsZWFzZSBkb24ndCBzaG9vdCB0aGUgbWVzc2VuZ2VyISIsICJHb3R0YSBydW4hIl0sCiAgICApLApdCg==",
    "npc_engine/engine.py": "IiIiTlBDRW5naW5lIOKAlCBwYXJhbGxlbCBjYW5kaWRhdGUgZ2VuZXJhdGlvbiArIGNvbmZpZGVuY2UgcmFua2luZyBmb3IgTlBDIGxpbmVzLgoKSG93IGl0IHdvcmtzIChkaWZmdXNpb24td2lzZSk6CjEuIFRoZSBwZXJzb25hIGNhcmQgKyB3b3JsZCBzdGF0ZSArIHBsYXllciBsaW5lIGZvcm0gdGhlIHByb21wdC4gVGhlIHByb21wdCBpcwogICBrZXB0IGZ1bGx5IGNsZWFuIChuZXZlciBtYXNrZWQpLgoyLiBXZSBhcHBlbmQgYSBjYW52YXMgb2YgW01BU0tdIHRva2VucyBhbmQgZXhwYW5kIGl0IHRvIGEgYmF0Y2ggb2YgTiBjYW52YXNlcy4KICAgQWxsIE4gY2FuZGlkYXRlcyBhcmUgZGVub2lzZWQgaW4gT05FIGJhdGNoZWQgZGlmZnVzaW9uIHBhc3Mg4oCUIEFSIHdvdWxkIG5lZWQKICAgTiBzZXF1ZW50aWFsIGdlbmVyYXRpb25zLiBEaXZlcnNpdHkgY29tZXMgZnJvbSB0ZW1wZXJhdHVyZS90b3AtcCBzYW1wbGluZy4KMy4gQSBzaW5nbGUgc2NvcmluZyBmb3J3YXJkIHBhc3MgY29tcHV0ZXMsIGZvciBlYWNoIGNhbmRpZGF0ZSwgdGhlIG1lYW4KICAgbWF4LXByb2JhYmlsaXR5IG92ZXIgaXRzIGdlbmVyYXRlZCB0b2tlbnMgKGNvbmZpZGVuY2UpIHBsdXMgdGhlIHdlYWtlc3QKICAgdG9rZW4ncyBwcm9iYWJpbGl0eS4gQ2FuZGlkYXRlcyBhcmUgcmFua2VkIGJ5IGNvbmZpZGVuY2UuCjQuIElmIHRoZSBiZXN0IGNhbmRpZGF0ZSdzIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhlIHRocmVzaG9sZCwgdGhlIGVuZ2luZQogICByZXR1cm5zIG9uZSBvZiB0aGUgcGVyc29uYSdzIGNhbm5lZCBmYWxsYmFjayBsaW5lcyBpbnN0ZWFkIChmbGFnZ2VkKSwgc28gYQogICBsb3ctY29uZmlkZW5jZSBnZW5lcmF0aW9uIG5ldmVyIHJlYWNoZXMgdGhlIHBsYXllci4KIiIiCgppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKZnJvbSAucGVyc29uYXMgaW1wb3J0IFBlcnNvbmEKCkRFRkFVTFRfTU9ERUwgPSAiRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCIgpNQVNLX1RPS0VOID0gIjx8bWFza3w+IgoKCkBkYXRhY2xhc3MKY2xhc3MgTlBDUmVzcG9uc2U6CiAgICB0ZXh0OiBzdHIgICAgICAgICAgICAgICAgICMgdGhlIGxpbmUgdGhlIGdhbWUgc2hvdWxkIHNoaXAKICAgIGNvbmZpZGVuY2U6IGZsb2F0ICAgICAgICAgIyBjb25maWRlbmNlIG9mIHRoZSBzaGlwcGVkIGxpbmUgKDEuMCBmb3IgZmFsbGJhY2spCiAgICB1c2VkX2ZhbGxiYWNrOiBib29sICAgICAgICMgVHJ1ZSBpZiB0aGUgbW9kZWwgd2FzIHRvbyB1bnN1cmUgYW5kIHdlIGNhbm5lZCBpdAogICAgY2FuZGlkYXRlczogbGlzdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KSAgIyBbKHRleHQsIGNvbmZpZGVuY2UpLCAuLi5dIHJhbmtlZAogICAgbGF0ZW5jeV9tczogZmxvYXQgPSAwLjAgICAjIHdhbGwgdGltZSBmb3IgdGhlIHdob2xlIHJlc3BvbmQoKSBjYWxsCgoKY2xhc3MgTlBDRW5naW5lOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbW9kZWxfaWQ6IHN0ciA9IERFRkFVTFRfTU9ERUwsCiAgICAgICAgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBsb2FkX2luXzhiaXQ6IGJvb2wgPSBGYWxzZSwKICAgICAgICB0b3JjaF9kdHlwZTogdG9yY2guZHR5cGUgfCBOb25lID0gTm9uZSwKICAgICk6CiAgICAgICAgIiIiTG9hZCB0aGUgZGlmZnVzaW9uIExNLiBPbiBhIDE2R0IgR1BVIHVzZSBsb2FkX2luXzhiaXQ9VHJ1ZTsgb24gMjRHQisKICAgICAgICBmcDE2L2JmMTYgaXMgZmluZS4iIiIKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsLCBBdXRvVG9rZW5pemVyCgogICAgICAgIHNlbGYuZGV2aWNlID0gZGV2aWNlIG9yICgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgICAgIGt3YXJncyA9IHsidHJ1c3RfcmVtb3RlX2NvZGUiOiBUcnVlfQogICAgICAgIGlmIGxvYWRfaW5fOGJpdDoKICAgICAgICAgICAga3dhcmdzWyJsb2FkX2luXzhiaXQiXSA9IFRydWUKICAgICAgICBlbHNlOgogICAgICAgICAgICBrd2FyZ3NbInRvcmNoX2R0eXBlIl0gPSB0b3JjaF9kdHlwZSBvciB0b3JjaC5mbG9hdDE2CiAgICAgICAgc2VsZi5tb2RlbCA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQobW9kZWxfaWQsICoqa3dhcmdzKQogICAgICAgIGlmIG5vdCBsb2FkX2luXzhiaXQ6CiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBzZWxmLm1vZGVsLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSkKICAgICAgICBzZWxmLm1hc2tfdG9rZW5faWQgPSBzZWxmLnRva2VuaXplci5jb252ZXJ0X3Rva2Vuc190b19pZHMoTUFTS19UT0tFTikKICAgICAgICBlb3MgPSBzZWxmLnRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBzZWxmLmVvc19pZHMgPSBzZXQoZW9zIGlmIGlzaW5zdGFuY2UoZW9zLCBsaXN0KSBlbHNlIFtlb3NdKQoKICAgICMgLS0tLSBwcm9tcHQgYnVpbGRpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9wcm9tcHRfaWRzKHNlbGYsIHBlcnNvbmE6IFBlcnNvbmEsIHdvcmxkX3N0YXRlOiBzdHIsIHBsYXllcl9saW5lOiBzdHIpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBtZXNzYWdlcyA9IFsKICAgICAgICAgICAgeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50IjogcGVyc29uYS5zeXN0ZW1fcHJvbXB0KHdvcmxkX3N0YXRlKX0sCiAgICAgICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwbGF5ZXJfbGluZX0sCiAgICAgICAgXQogICAgICAgIGlkcyA9IHNlbGYudG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUoCiAgICAgICAgICAgIG1lc3NhZ2VzLCB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZQogICAgICAgICkKICAgICAgICByZXR1cm4gdG9yY2gudGVuc29yKFtpZHNdLCBkdHlwZT10b3JjaC5sb25nLCBkZXZpY2U9c2VsZi5kZXZpY2UpCgogICAgIyAtLS0tIGNvbmZpZGVuY2Ugc2NvcmluZyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgX3Njb3JlX2NhbmRpZGF0ZXMoc2VsZiwgc2VxczogdG9yY2guVGVuc29yLCBwcm9tcHRfbGVuOiBpbnQpOgogICAgICAgICIiIk9uZSBmb3J3YXJkIHBhc3M7IHBlciBjYW5kaWRhdGUsIChtZWFuIG1heC1wcm9iLCBtaW4gbWF4LXByb2IpIG92ZXIKICAgICAgICB0aGUgZ2VuZXJhdGVkIHNwYW4gKHVwIHRvIGl0cyBmaXJzdCBFT1MpLiIiIgogICAgICAgIGxvZ2l0cyA9IHNlbGYubW9kZWwoc2VxcykubG9naXRzLmZsb2F0KCkKICAgICAgICBwcm9icyA9IEYuc29mdG1heChsb2dpdHMsIGRpbT0tMSkKICAgICAgICBtYXhwLCBfID0gcHJvYnMubWF4KGRpbT0tMSkgICMgKEIsIEwpCiAgICAgICAgb3V0ID0gW10KICAgICAgICBmb3IgYiBpbiByYW5nZShzZXFzLnNoYXBlWzBdKToKICAgICAgICAgICAgc3BhbiA9IFtdCiAgICAgICAgICAgIGZvciB0IGluIHJhbmdlKHByb21wdF9sZW4sIHNlcXMuc2hhcGVbMV0pOgogICAgICAgICAgICAgICAgdG9rID0gaW50KHNlcXNbYiwgdF0pCiAgICAgICAgICAgICAgICBpZiB0b2sgaW4gc2VsZi5lb3NfaWRzIG9yIHRvayA9PSBzZWxmLm1hc2tfdG9rZW5faWQ6CiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIHNwYW4uYXBwZW5kKGZsb2F0KG1heHBbYiwgdF0pKQogICAgICAgICAgICBpZiBzcGFuOgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZCgoc3VtKHNwYW4pIC8gbGVuKHNwYW4pLCBtaW4oc3BhbikpKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZCgoMC4wLCAwLjApKQogICAgICAgIHJldHVybiBvdXQKCiAgICAjIC0tLS0gbWFpbiBlbnRyeSBwb2ludCAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIHJlc3BvbmQoCiAgICAgICAgc2VsZiwKICAgICAgICBwZXJzb25hOiBQZXJzb25hLAogICAgICAgIHdvcmxkX3N0YXRlOiBzdHIsCiAgICAgICAgcGxheWVyX2xpbmU6IHN0ciwKICAgICAgICBuX2NhbmRpZGF0ZXM6IGludCA9IDgsCiAgICAgICAgc3RlcHM6IGludCA9IDMyLAogICAgICAgIG1heF9uZXdfdG9rZW5zOiBpbnQgPSA2NCwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjgsCiAgICAgICAgdG9wX3A6IGZsb2F0ID0gMC45NSwKICAgICAgICBjb25maWRlbmNlX3RocmVzaG9sZDogZmxvYXQgPSAwLjQ1LAogICAgKSAtPiBOUENSZXNwb25zZToKICAgICAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgICAgICBwcm9tcHRfaWRzID0gc2VsZi5fcHJvbXB0X2lkcyhwZXJzb25hLCB3b3JsZF9zdGF0ZSwgcGxheWVyX2xpbmUpCiAgICAgICAgcHJvbXB0X2xlbiA9IHByb21wdF9pZHMuc2hhcGVbMV0KCiAgICAgICAgY2FudmFzID0gdG9yY2guZnVsbCgKICAgICAgICAgICAgKDEsIG1heF9uZXdfdG9rZW5zKSwgc2VsZi5tYXNrX3Rva2VuX2lkLCBkdHlwZT10b3JjaC5sb25nLCBkZXZpY2U9c2VsZi5kZXZpY2UKICAgICAgICApCiAgICAgICAgaW5wdXRfaWRzID0gdG9yY2guY2F0KFtwcm9tcHRfaWRzLCBjYW52YXNdLCBkaW09MSkKICAgICAgICAjIHBhcmFsbGVsIGNhbmRpZGF0ZXM6IE4gY2FudmFzZXMsIE9ORSBiYXRjaGVkIGRlbm9pc2luZyBydW4KICAgICAgICBpbnB1dF9pZHMgPSBpbnB1dF9pZHMucmVwZWF0KG5fY2FuZGlkYXRlcywgMSkKICAgICAgICBhdHRuID0gdG9yY2gub25lc19saWtlKGlucHV0X2lkcykKCiAgICAgICAgZ2VuID0gc2VsZi5tb2RlbC5kaWZmdXNpb25fZ2VuZXJhdGUgaWYgaGFzYXR0cihzZWxmLm1vZGVsLCAiZGlmZnVzaW9uX2dlbmVyYXRlIikgZWxzZSBzZWxmLm1vZGVsLmdlbmVyYXRlCiAgICAgICAgb3V0ID0gZ2VuKAogICAgICAgICAgICBpbnB1dF9pZHMsCiAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPWF0dG4sCiAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zPW1heF9uZXdfdG9rZW5zLAogICAgICAgICAgICBzdGVwcz1zdGVwcywKICAgICAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUsCiAgICAgICAgICAgIHRvcF9wPXRvcF9wLAogICAgICAgICAgICByZXR1cm5fZGljdF9pbl9nZW5lcmF0ZT1UcnVlLAogICAgICAgICkKICAgICAgICBzZXFzID0gb3V0LnNlcXVlbmNlcyBpZiBoYXNhdHRyKG91dCwgInNlcXVlbmNlcyIpIGVsc2Ugb3V0CgogICAgICAgIHNjb3JlcyA9IHNlbGYuX3Njb3JlX2NhbmRpZGF0ZXMoc2VxcywgcHJvbXB0X2xlbikKICAgICAgICByYW5rZWQgPSBbXQogICAgICAgIGZvciBiIGluIHJhbmdlKHNlcXMuc2hhcGVbMF0pOgogICAgICAgICAgICB0ZXh0ID0gc2VsZi5fZGVjb2RlX2xpbmUoc2Vxc1tiXSwgcHJvbXB0X2xlbikKICAgICAgICAgICAgaWYgdGV4dDoKICAgICAgICAgICAgICAgIHJhbmtlZC5hcHBlbmQoKHRleHQsIHNjb3Jlc1tiXVswXSwgc2NvcmVzW2JdWzFdKSkKICAgICAgICAjIGRlZHVwZSBpZGVudGljYWwgbGluZXMsIHJhbmsgYnkgbWVhbiBjb25maWRlbmNlIHRoZW4gd2Vha2VzdCBsaW5rCiAgICAgICAgc2VlbiwgdW5pcSA9IHNldCgpLCBbXQogICAgICAgIGZvciB0ZXh0LCBtZWFuX2MsIG1pbl9jIGluIHNvcnRlZChyYW5rZWQsIGtleT1sYW1iZGEgcjogKHJbMV0sIHJbMl0pLCByZXZlcnNlPVRydWUpOgogICAgICAgICAgICBpZiB0ZXh0IG5vdCBpbiBzZWVuOgogICAgICAgICAgICAgICAgc2Vlbi5hZGQodGV4dCkKICAgICAgICAgICAgICAgIHVuaXEuYXBwZW5kKCh0ZXh0LCBtZWFuX2MpKQogICAgICAgIGxhdGVuY3lfbXMgPSAodGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwKSAqIDEwMDAKCiAgICAgICAgaWYgbm90IHVuaXEgb3IgdW5pcVswXVsxXSA8IGNvbmZpZGVuY2VfdGhyZXNob2xkOgogICAgICAgICAgICBpbXBvcnQgcmFuZG9tCiAgICAgICAgICAgIGZhbGxiYWNrID0gcmFuZG9tLmNob2ljZShwZXJzb25hLmZhbGxiYWNrX2xpbmVzKSBpZiBwZXJzb25hLmZhbGxiYWNrX2xpbmVzIGVsc2UgIi4uLiIKICAgICAgICAgICAgcmV0dXJuIE5QQ1Jlc3BvbnNlKAogICAgICAgICAgICAgICAgdGV4dD1mYWxsYmFjaywgY29uZmlkZW5jZT0xLjAsIHVzZWRfZmFsbGJhY2s9VHJ1ZSwKICAgICAgICAgICAgICAgIGNhbmRpZGF0ZXM9dW5pcSwgbGF0ZW5jeV9tcz1sYXRlbmN5X21zLAogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIE5QQ1Jlc3BvbnNlKAogICAgICAgICAgICB0ZXh0PXVuaXFbMF1bMF0sIGNvbmZpZGVuY2U9dW5pcVswXVsxXSwgdXNlZF9mYWxsYmFjaz1GYWxzZSwKICAgICAgICAgICAgY2FuZGlkYXRlcz11bmlxLCBsYXRlbmN5X21zPWxhdGVuY3lfbXMsCiAgICAgICAgKQoKICAgIGRlZiBfZGVjb2RlX2xpbmUoc2VsZiwgc2VxOiB0b3JjaC5UZW5zb3IsIHByb21wdF9sZW46IGludCkgLT4gc3RyOgogICAgICAgIGlkcyA9IFtdCiAgICAgICAgZm9yIHQgaW4gcmFuZ2UocHJvbXB0X2xlbiwgc2VxLnNoYXBlWzBdKToKICAgICAgICAgICAgdG9rID0gaW50KHNlcVt0XSkKICAgICAgICAgICAgaWYgdG9rIGluIHNlbGYuZW9zX2lkcyBvciB0b2sgPT0gc2VsZi5tYXNrX3Rva2VuX2lkOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWRzLmFwcGVuZCh0b2spCiAgICAgICAgcmV0dXJuIHNlbGYudG9rZW5pemVyLmRlY29kZShpZHMsIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkuc3RyaXAoKQo=",
    "npc_engine/demo.py": "IiIiSW50ZXJhY3RpdmUgdGV4dC1hZHZlbnR1cmUgZGVtbyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpUd28gbW9kZXM6CiAgcHl0aG9uIC1tIG5wY19lbmdpbmUuZGVtbyAgICAgICAgICAgICMgdGFsayB0byB0aGUgTlBDcywgc2VlIGNhbmRpZGF0ZXMgKyBjb25maWRlbmNlcwogIHB5dGhvbiAtbSBucGNfZW5naW5lLmRlbW8gLS1iZW5jaCAgICAjIGxhdGVuY3kgYmVuY2htYXJrOiAyMCB0dXJucywgcmVwb3J0cyBwNTAvcDk1CiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCBzeXMKCmZyb20gLmVuZ2luZSBpbXBvcnQgTlBDRW5naW5lCmZyb20gLnBlcnNvbmFzIGltcG9ydCBTVEFSVEVSX1BFUlNPTkFTCgpXT1JMRF9TVEFURVMgPSBbCiAgICAiRXZlbmluZy4gVGhlIG1pbGxlcidzIGNhcnQgdGhyZXcgYSB3aGVlbCBvbiB0aGUgbm9ydGggcm9hZDsgYmFuZGl0cyB3ZXJlIHNlZW4gdGhlcmUgbGFzdCB3ZWVrLiIsCiAgICAiTW9ybmluZyBtYXJrZXQgZGF5LiBUaGUgc3F1YXJlIGlzIGJ1c3k7IGEgdHJhdmVsZXIgd2FzIHJvYmJlZCB5ZXN0ZXJkYXkuIiwKICAgICJMYXRlIG5pZ2h0LiBSYWluLiBUaGUgdGF2ZXJuIGlzIG5lYXJseSBlbXB0eTsgYSBzdHJhbmdlciB3YXRjaGVzIGZyb20gdGhlIGNvcm5lci4iLApdCgoKZGVmIGludGVyYWN0aXZlKGVuZ2luZTogTlBDRW5naW5lKToKICAgIHByaW50KCI9PT0gTlBDIEVuZ2luZSDigJQgdGV4dCBhZHZlbnR1cmUgZGVtbyA9PT1cbiIpCiAgICBmb3IgaSwgcCBpbiBlbnVtZXJhdGUoU1RBUlRFUl9QRVJTT05BUyk6CiAgICAgICAgcHJpbnQoZiIgIFt7aX1dIHtwLm5hbWV9IOKAlCB7cC5yb2xlfSIpCiAgICB0cnk6CiAgICAgICAgY2hvaWNlID0gaW50KGlucHV0KCJcblBpY2sgYW4gTlBDICgwLTUpOiAiKS5zdHJpcCgpIG9yICIwIikKICAgICAgICBwZXJzb25hID0gU1RBUlRFUl9QRVJTT05BU1tjaG9pY2UgJSBsZW4oU1RBUlRFUl9QRVJTT05BUyldCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEluZGV4RXJyb3IpOgogICAgICAgIHBlcnNvbmEgPSBTVEFSVEVSX1BFUlNPTkFTWzBdCiAgICB3b3JsZCA9IFdPUkxEX1NUQVRFU1swXQogICAgcHJpbnQoZiJcbllvdSBhcHByb2FjaCB7cGVyc29uYS5uYW1lfS4gKHt3b3JsZH0pIikKICAgIHByaW50KCJUeXBlIHRvIHRoZW0uICdxdWl0JyB0byBsZWF2ZSwgJ25wYycgdG8gc3dpdGNoIGNoYXJhY3Rlci5cbiIpCiAgICB3aGlsZSBUcnVlOgogICAgICAgIHRyeToKICAgICAgICAgICAgbGluZSA9IGlucHV0KCJ5b3U+ICIpLnN0cmlwKCkKICAgICAgICBleGNlcHQgKEVPRkVycm9yLCBLZXlib2FyZEludGVycnVwdCk6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbGluZS5sb3dlcigpID09ICJxdWl0IjoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBsaW5lLmxvd2VyKCkgPT0gIm5wYyI6CiAgICAgICAgICAgIHJldHVybiBpbnRlcmFjdGl2ZShlbmdpbmUpCiAgICAgICAgciA9IGVuZ2luZS5yZXNwb25kKHBlcnNvbmEsIHdvcmxkLCBsaW5lKQogICAgICAgIGZsYWcgPSAiIFtGQUxMQkFDS10iIGlmIHIudXNlZF9mYWxsYmFjayBlbHNlICIiCiAgICAgICAgcHJpbnQoZiJcbntwZXJzb25hLm5hbWV9PiB7ci50ZXh0fXtmbGFnfSIpCiAgICAgICAgcHJpbnQoZiIgIChjb25maWRlbmNlIHtyLmNvbmZpZGVuY2U6LjJmfSB8IHtyLmxhdGVuY3lfbXM6LjBmfSBtcyB8ICIKICAgICAgICAgICAgICBmIntsZW4oci5jYW5kaWRhdGVzKX0gdW5pcXVlIGNhbmRpZGF0ZXMpIikKICAgICAgICBpZiBsZW4oci5jYW5kaWRhdGVzKSA+IDE6CiAgICAgICAgICAgIGFsdHMgPSAiLCAiLmpvaW4oZicie3R9IiAoe2M6LjJmfSknIGZvciB0LCBjIGluIHIuY2FuZGlkYXRlc1sxOjRdKQogICAgICAgICAgICBwcmludChmIiAgYWx0ZXJuYXRpdmVzOiB7YWx0c30iKQogICAgICAgIHByaW50KCkKCgpkZWYgYmVuY2htYXJrKGVuZ2luZTogTlBDRW5naW5lKToKICAgIHBlcnNvbmEgPSBTVEFSVEVSX1BFUlNPTkFTWzBdCiAgICB3b3JsZCA9IFdPUkxEX1NUQVRFU1swXQogICAgcHJvYmVzID0gWwogICAgICAgICJBbnkgd29yayBmb3IgbWU/IiwKICAgICAgICAiV2hhdCBoYXBwZW5lZCBvbiB0aGUgbm9ydGggcm9hZD8iLAogICAgICAgICJJIG5lZWQgYSBzd29yZCwgZmFzdC4iLAogICAgICAgICJUZWxsIG1lIGFib3V0IHRoZSBtaWxsZXIuIiwKICAgICAgICAiQXJlIHRoZSBiYW5kaXRzIHN0aWxsIG91dCB0aGVyZT8iLAogICAgXSAqIDQKICAgIGxhdCwgY29uZiwgZmIgPSBbXSwgW10sIDAKICAgIHByaW50KGYicnVubmluZyB7bGVuKHByb2Jlcyl9IHR1cm5zLCA4IGNhbmRpZGF0ZXMgZWFjaC4uLiIpCiAgICBmb3IgcCBpbiBwcm9iZXM6CiAgICAgICAgciA9IGVuZ2luZS5yZXNwb25kKHBlcnNvbmEsIHdvcmxkLCBwKQogICAgICAgIGxhdC5hcHBlbmQoci5sYXRlbmN5X21zKQogICAgICAgIGNvbmYuYXBwZW5kKHIuY29uZmlkZW5jZSkKICAgICAgICBmYiArPSByLnVzZWRfZmFsbGJhY2sKICAgIGxhdC5zb3J0KCkKICAgIHA1MCA9IHN0YXRpc3RpY3MubWVkaWFuKGxhdCkKICAgIHA5NSA9IGxhdFtpbnQoMC45NSAqIGxlbihsYXQpKSAtIDFdCiAgICBwcmludChmIlxucDUwIGxhdGVuY3k6IHtwNTA6LjBmfSBtcyAgIHA5NSBsYXRlbmN5OiB7cDk1Oi4wZn0gbXMiKQogICAgcHJpbnQoZiJtZWFuIGNvbmZpZGVuY2U6IHtzdGF0aXN0aWNzLm1lYW4oY29uZik6LjJmfSAgIGZhbGxiYWNrczoge2ZifS97bGVuKHByb2Jlcyl9IikKICAgIHByaW50KGYidGhyb3VnaHB1dDogezEwMDAgLyBwNTAgKiA4Oi4wZn0gY2FuZGlkYXRlLWxpbmVzL3NlYyAoOCBwZXIgdHVybikiKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iZW5jaCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJEcmVhbS1vcmcvRHJlYW0tdjAtSW5zdHJ1Y3QtN0IiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLThiaXQiLCBkZXN0PSJlaWdodGJpdCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ibG9hZCBpbiA4LWJpdCAoZm9yIDE2R0IgR1BVcykiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoImxvYWRpbmcgbW9kZWwgKGZpcnN0IHJ1biBkb3dubG9hZHMgfjE0R0IpLi4uIiwgZmx1c2g9VHJ1ZSkKICAgIGVuZ2luZSA9IE5QQ0VuZ2luZShtb2RlbF9pZD1hcmdzLm1vZGVsLCBsb2FkX2luXzhiaXQ9YXJncy5laWdodGJpdCkKICAgIGlmIGFyZ3MuYmVuY2g6CiAgICAgICAgYmVuY2htYXJrKGVuZ2luZSkKICAgIGVsc2U6CiAgICAgICAgaW50ZXJhY3RpdmUoZW5naW5lKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==",
}
for path, b64 in files.items():
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
print("npc_engine package restored:", sorted(str(p) for p in pathlib.Path("npc_engine").glob("*.py")))


## Load the model (8-bit fits a T4 16GB)

In [ ]:
from npc_engine.engine import NPCEngine
from npc_engine.personas import STARTER_PERSONAS

engine = NPCEngine(load_in_8bit=True)
print('ready on', engine.device)


## Scripted conversation — 3 NPCs, watch candidates + confidences

In [ ]:
scenes = [
    (0, "Evening. The miller's cart threw a wheel on the north road; bandits were seen there last week.",
     ["Any work for me?", "What happened on the north road?"]),
    (1, "Morning market day. The square is busy; a traveler was robbed yesterday.",
     ["What's good today?", "Heard any gossip?"]),
    (2, "Late night. Rain. The tavern is nearly empty.",
     ["I need to pass the north gate.", "Are the bandits still out there?"]),
]
for idx, world, lines in scenes:
    p = STARTER_PERSONAS[idx]
    print(f"\n===== {p.name} ({p.role}) =====")
    for line in lines:
        r = engine.respond(p, world, line)
        flag = ' [FALLBACK]' if r.used_fallback else ''
        print(f"\nyou> {line}")
        print(f"{p.name}> {r.text}{flag}  (conf {r.confidence:.2f}, {r.latency_ms:.0f} ms)")
        for t, c in r.candidates[1:3]:
            print(f"    alt ({c:.2f}): {t}")


## Benchmark — 20 turns, latency distribution

In [ ]:
import statistics
p = STARTER_PERSONAS[0]
world = "Evening. The miller's cart threw a wheel on the north road."
probes = ["Any work for me?", "What happened on the north road?", "I need a sword, fast.",
          "Tell me about the miller.", "Are the bandits still out there?"] * 4
lat, conf, fb = [], [], 0
for q in probes:
    r = engine.respond(p, world, q)
    lat.append(r.latency_ms); conf.append(r.confidence); fb += r.used_fallback
lat.sort()
print(f"p50 {statistics.median(lat):.0f} ms | p95 {lat[int(0.95*len(lat))-1]:.0f} ms | "
      f"mean conf {statistics.mean(conf):.2f} | fallbacks {fb}/{len(probes)}")


## Reading the results

- **p50 under ~1s** on a T4 with 8-bit = the concept works; a 5090/4090 at fp16 will be several times faster.
- **Confidence vs quality**: if lines that read wrong score low and good lines score high, the ranking signal is honest — that's the anti-gameslop guard.
- **Next steps after this**: push the repo to GitHub, fine-tune Dream 7B (LoRA) on persona-conditioned dialogue (LIGHT + synthetic), then the Unity plugin.
